<a href="https://colab.research.google.com/github/Teomorales20/SenalesSistemas/blob/master/Taller_1_11_2025_TransformadaFourier_SAudio_Mateo_Morales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Filtrado espectral utilizando FFT con señal de audio

- Se utilizará la `fft` para filtrar una señal de audio.

- Se puede usar la API `youtube-dl` para descargar un video de youtube y extraer el audio en formato mp3.

In [ ]:
!python3 -m pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz

In [ ]:
!yt-dlp --extract-audio -o "audio" --audio-format mp3 https://youtu.be/k8nxe6UE1gY?si=kjvQKFPCFpTv3uqr

In [ ]:
!ffmpeg -y -i audio.mp3 output.wav

In [ ]:
!pip install soundfile

In [ ]:
from os import fsdecode
import soundfile as sf

nombre_out = "output.wav"
x, fs = sf.read(nombre_out)
print("Frecuencia de muestreo %.2F[Hz] %s" % (fs, nombre_out))


In [ ]:
x.shape #numero de muestras y numero de canales

In [ ]:
from IPython.display import Audio
ns = 30
Audio(x[:int(ns*fs),:].T, rate=fs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
xpro = x.copy() #copiar archivos para procesar, Crea una copia del audio original (x) para procesarlo sin modificar los datos originales.
ti = 10 #tiempo incio a procesar seg
tf = 15 #tiempo final a procesar seg
xs = xpro[int(ti*fs):int((tf*fs)),:] #tomo desde la muestra ti*fs hasta tf*fs sin incluirla, y tomo los dos canales

tt = np.arange(ti,tf,1/fs) # vector de tiempo, Va desde ti hasta tf, con incrementos de 1/fs segundos.
plt.plot(tt,xs)
plt.legend(('canal 1','canal 2'))
plt.xlabel('$t[s]$')
plt.ylabel('$x(t)$')
plt.legend()
plt.show()


#Gráficos separados por canal
# ============================================================
plt.figure(figsize=(10, 6))

# Canal 1
plt.subplot(2, 1, 1)
plt.plot(tt, xs[:, 0], color='tab:blue')
plt.title("Canal 1 (Izquierdo)")
plt.ylabel("$x_1(t)$")
plt.grid(True)

# Canal 2
plt.subplot(2, 1, 2)
plt.plot(tt, xs[:, 1], color='tab:orange')
plt.title("Canal 2 (Derecho)")
plt.xlabel("$t\,[s]$")
plt.ylabel("$x_2(t)$")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 🎵 Análisis espectral del fragmento de audio
# ============================================================

# ------------------------------------------------------------
# 1️⃣ Transformada de Fourier de ambos canales
# ------------------------------------------------------------
# Se aplica la Transformada Rápida de Fourier (FFT) real a cada canal del audio.
# El parámetro axis=0 indica que la FFT se calcula a lo largo de las filas
# (es decir, sobre el eje temporal), para cada columna (canal) de la matriz xs.
# rfft() se usa en lugar de fft() porque la señal es real, y solo se necesitan
# las frecuencias positivas (evita duplicar la información conjugada).
Xw = np.fft.rfft(xs, axis=0)

# ------------------------------------------------------------
# 2️⃣ Cálculo del vector de frecuencias
# ------------------------------------------------------------
# Se genera el eje de frecuencias (en Hz) correspondiente a cada punto del espectro.
# np.size(xs,0) devuelve el número de muestras del fragmento (filas de xs),
# y 1/fs indica el paso temporal entre cada muestra.
# En resumen, vf contiene las frecuencias donde está definida la FFT.
vf = np.fft.rfftfreq(np.size(xs, 0), 1/fs)

# ------------------------------------------------------------
# 3️⃣ Visualización del espectro de magnitud
# ------------------------------------------------------------
# Se grafica la magnitud (valor absoluto) del espectro en función de la frecuencia.
# abs(Xw) devuelve la amplitud espectral de cada frecuencia, para ambos canales.
# Como Xw tiene dos columnas (canal izquierdo y canal derecho), se trazan dos curvas.
plt.plot(vf, abs(Xw))

# ------------------------------------------------------------
# 4️⃣ Etiquetas y formato del gráfico
# ------------------------------------------------------------
# Se agregan las leyendas, título y etiquetas con formato LaTeX para mayor claridad.
plt.legend(('canal 1', 'canal 2'))
plt.title(r'Espectro audio original')
plt.xlabel(r'$f[Hz]$', fontsize=14)
plt.ylabel(r'$|X[n]|$', fontsize=14)
plt.grid(True)

# ------------------------------------------------------------
# 5️⃣ Mostrar la figura
# ------------------------------------------------------------
# Finalmente, se renderiza la gráfica con el espectro de frecuencias del fragmento
# de audio seleccionado. En ella se pueden observar los armónicos y la energía
# distribuida a lo largo del rango de frecuencias de ambos canales.
plt.show()

# Tipos de filtrado

- Según el rango de frecuencias a filtrar (según el diagrama de magnitud), los filtros se pueden clasificar en:

  - Filtro pasa-bajas (low-pass)
  - Filtro pasa-altas (high-pass)
  - Filtro pasa-banda (band-pass)
  - Filtro rechaza banda (band-stop)

  ![filtros](https://github.com/amalvarezme/SenalesSistemas/blob/master/3_SerieyTransformadaFourier/filters2.jpg?raw=1)


- En este caso, se plantea un filtro pasa banda, apagando los armónicos que no nos interesa.

In [ ]:
#filtrar espectro
Xwf = Xw.copy()
f1 = 100 #frecuencia en Hz corte 1
f2 = 3400 #frecuencia en Hz corte 2
ind = ~((vf > f1) & (vf < f2)) #frecuencias eliminar-> recueder que ~ actua como negación
Xwf[ind,:] = 0
plt.plot(vf,abs(Xwf))
plt.legend(('canal 1','canal 2'))
plt.show()

- Ahora, se reconstruye la señal mediante la fft inversa.

In [ ]:
xe2 = np.fft.irfft(Xwf,axis=0)

In [ ]:
Audio(xe2[:int(fs*ns),:].T,rate=fs)

In [ ]:
Fo = 5000
F2 = 10000
Fs = 50000
ti = 0
tf = 5
tv = np.arange(ti,tf,1/Fs)
A = 20
xt = A*np.cos(2*np.pi*Fo*tv) +0.5*A*np.cos(2*np.pi*F2*tv)
plt.plot(tv,xt)
plt.show()

In [ ]:
Audio(xt,rate=Fs)#repoducir señal filtrada

In [ ]:
Xwc = np.fft.rfft(xt) # axis=0 permite aplicar fft por cada columna de xpro
vfc = np.fft.rfftfreq(xt.shape[0],1/Fs)

plt.plot(vfc,abs(Xwc))#se grafica la magnitud
plt.title(r'Espectro cos')
plt.xlabel(r'$f[Hz]$',fontsize = 14)
plt.ylabel(r'$|X[n]|$',fontsize = 14)
plt.show()

#Tipos de filtrado


La frecuencia de -3dB (también conocida como frecuencia de esquina o frecuencia de corte a medio potencia) es un concepto clave en el análisis de señales y el diseño de filtros. Se refiere a la frecuencia a la que la potencia de la señal de salida de un sistema (como un filtro) ha disminuido a la mitad de su potencia máxima. En una escala logarítmica en decibelios (dB), una disminución a la mitad de la potencia corresponde a una caída de 3 dB (10 * log10(0.5) ≈ -3.01 dB).

En el contexto del filtrado espectral de señales, la frecuencia de -3dB es comúnmente utilizada para definir la "frecuencia de corte" de un filtro. La frecuencia de corte marca la transición entre las frecuencias que el filtro permite pasar (banda de paso) y las frecuencias que el filtro atenúa (banda de atenuación).

La relación entre la frecuencia de -3dB y las frecuencias de corte depende del tipo de filtro:

1. Filtro Pasa-Bajas
2. Filtro Pasa-Altas
3. Filtro Pasa-Banda
4. Filtro Rechaza-Banda

- Implementación de un filtro pasa bajas, un pasa altas, un pasa bandas, y un rechaza bandas utilizando la FFT y la iFFT sobre 5 segundos de una canción de YouTube.

In [ ]:
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

# Se debe incluir el link del video a procesar:
link="https://www.youtube.com/watch?v=YkBrEfFpnBs" # <--- Change this to your new link
!yt-dlp --extract-audio -o "audio" --audio-format mp3 {link}

!python3 -m pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz
!ffmpeg -y -i audio.mp3 output.wav
!pip install soundfile

import soundfile as sf # para instalar pip install soundfile
#lee archivos wav
nombre_out = "output.wav"
x, fs = sf.read(nombre_out)
# read speech signal from file
print('Frecuencia de muestreo %.2f[Hz]\naudio %s' % (fs,nombre_out))


# Seleccionar 5 segundos de audio (ajusta el inicio si es necesario)
ns = 5
x_segment = x[:int(fs * ns), :]

# Aplicar la FFT
Xw = np.fft.rfft(x_segment, axis=0)
vf = np.fft.rfftfreq(np.size(x_segment, 0), 1/fs)

# --- Implementación de filtros ---

# Filtro Pasa Bajas
print("Aplicando filtro Pasa Bajas...")
Xwf_lp = Xw.copy()
f_cut_lp = 500 # Frecuencia de corte en Hz
ind_lp = (vf > f_cut_lp)
Xwf_lp[ind_lp, :] = 0
xe2_lp = np.fft.irfft(Xwf_lp, axis=0)
plt.plot(vf, abs(Xwf_lp))
plt.title(f'Espectro con Filtro Pasa Bajas (Corte: {f_cut_lp} Hz)')
plt.xlabel(r'$f[Hz]$', fontsize=14)
plt.ylabel(r'$|X[n]|$', fontsize=14)
plt.legend(('canal 1', 'canal 2'))
plt.show()
print("Audio filtrado Pasa Bajas:")
display(Audio(xe2_lp.T, rate=fs))

# Filtro Pasa Altas
print("Aplicando filtro Pasa Altas...")
Xwf_hp = Xw.copy()
f_cut_hp = 1000 # Frecuencia de corte en Hz
ind_hp = (vf < f_cut_hp)
Xwf_hp[ind_hp, :] = 0
xe2_hp = np.fft.irfft(Xwf_hp, axis=0)
plt.plot(vf, abs(Xwf_hp))
plt.title(f'Espectro con Filtro Pasa Altas (Corte: {f_cut_hp} Hz)')
plt.xlabel(r'$f[Hz]$', fontsize=14)
plt.ylabel(r'$|X[n]|$', fontsize=14)
plt.legend(('canal 1', 'canal 2'))
plt.show()
print("Audio filtrado Pasa Altas:")
display(Audio(xe2_hp.T, rate=fs))

# Filtro Pasa Banda
print("Aplicando filtro Pasa Banda...")
Xwf_bp = Xw.copy()
f1_bp = 300  # Frecuencia de corte inferior en Hz
f2_bp = 2000 # Frecuencia de corte superior en Hz
ind_bp = ~((vf > f1_bp) & (vf < f2_bp))
Xwf_bp[ind_bp, :] = 0
xe2_bp = np.fft.irfft(Xwf_bp, axis=0)
plt.plot(vf, abs(Xwf_bp))
plt.title(f'Espectro con Filtro Pasa Banda ({f1_bp}-{f2_bp} Hz)')
plt.xlabel(r'$f[Hz]$', fontsize=14)
plt.ylabel(r'$|X[n]|$', fontsize=14)
plt.legend(('canal 1', 'canal 2'))
plt.show()
print("Audio filtrado Pasa Banda:")
display(Audio(xe2_bp.T, rate=fs))

# Filtro Rechaza Banda
print("Aplicando filtro Rechaza Banda...")
Xwf_bs = Xw.copy()
f1_bs = 800  # Frecuencia de corte inferior de la banda a rechazar en Hz
f2_bs = 1500 # Frecuencia de corte superior de la banda a rechazar en Hz
ind_bs = ((vf > f1_bs) & (vf < f2_bs))
Xwf_bs[ind_bs, :] = 0
xe2_bs = np.fft.irfft(Xwf_bs, axis=0)
plt.plot(vf, abs(Xwf_bs))
plt.title(f'Espectro con Filtro Rechaza Banda ({f1_bs}-{f2_bs} Hz)')
plt.xlabel(r'$f[Hz]$', fontsize=14)
plt.ylabel(r'$|X[n]|$', fontsize=14)
plt.legend(('canal 1', 'canal 2'))
plt.show()
print("Audio filtrado Rechaza Banda:")
display(Audio(xe2_bs.T, rate=fs))